## Step 1: Set up directory tree, extract visibilities, carry out imaging

Import packages, define addresses within local machine

In [1]:
import numpy as np
import os
import pickle

### This is the address on your local machine to the package
miaopath='/Users/jonty/Documents/GitHub/miao'
### This is the address on your local machine to CASA
casapath='/Applications/CASA.app/Contents/MacOS/'
### This is the alias for CARTA
carta='/Applications/CARTA.app/Contents/MacOS/CARTA'
### This is the name of the folder where you want to carry out all the analysis and modelling - typically
### the name of the source you observed
sourcetag='hd377'
### And this is the address on your local machine where that folder will be created (if it doesn't exist already)
workingdir='/Users/jonty/mydata/aca_disks/'
### Finally, this is the address on your local machine of your CASA visibility MS file. This can be a list of multiple
### files if you want to image and model visibilities separately (e.g. if you had different pointings in a mosaic,
### different observation dates, different arrays, etc.)
vis=['/Users/jonty/mydata/aca_disks/hd377/calibratedms/hd377_calibrated_merged.ms']

#Create directories
nvis=len(vis)
if not os.path.exists(workingdir+'/'+sourcetag):
    os.mkdir(workingdir+'/'+sourcetag)
    print('Creating directory for object '+sourcetag+' at '+workingdir)
else:
    print('Directory for object '+sourcetag+' already exists at '+workingdir)

Directory for object hd377 already exists at /Users/jonty/mydata/aca_disks/


Decide on parameters that you want for CASA imaging through tclean. See CASA tclean documentation for more info.
By default, we will produce images with pixel and image sizes chosen according to the u-v distances covered in the dataset (see later). Multiscale deconvolution will be used.

In [2]:
### Is your observation a mosaic? (i.e. does it contain multiple pointings, or multiple heterogeneous arrays (e.g.
### ALMA 12m + ACA 7m)?
mosaic=False
if mosaic:
    mosaic=True
    ### If observation is a mosaic, what coordinates do you need at the center of the image? 
    phasecenter='J2000 22h57m39.449700 -29d37m22.68681'
else:
    phasecenter=''
### What weighting scheme would you like?
weighting='natural'
if weighting=='briggs':
    robust='0.5' #needs to be a string
else:
    robust=''
### Would you like to add a u-v taper (practically 'smoothing' your data in image space)?
uvtaper=[''] #e.g. ['1arcsec']
### Do you want to interact with tclean? I.e. carry out deconvolution and draw clean masks interactively?
### NB if False, running imaging within CASA separately from this notebook is not needed.
### NBB BUT your images will just be dirty images, and no deconvolution will be carried out.
### casaviewer no longer supported on MacOS, set this to false, do cleaning external to these notebooks
interactive=False

Create directory structure for imaging, print imaging parameters chosen above

In [3]:
#Create directories needed for imaging
os.chdir(workingdir+'/'+sourcetag)
for i in ['calibratedms', 'imaging']:
    if not os.path.exists(workingdir+'/'+sourcetag+'/'+i):
         os.mkdir(workingdir+'/'+sourcetag+'/'+i)
!cp -r {miaopath}/utils/mstonumpyortxt_multiple.py {workingdir}/{sourcetag}/calibratedms/.
for i in np.arange(len(vis)):
    if not os.path.exists('calibratedms/'+vis[i].rsplit('/',1)[1]):
        !cp -r {vis[i]} {workingdir}/{sourcetag}/calibratedms/.
    vis[i]=vis[i].rsplit('/',1)[1]
    #print(vis[i])
!cp -r {miaopath}/utils/imagingscript_multiple.py {workingdir}/{sourcetag}/imaging/. 

#Save directories and dataset names
pickle.dump([miaopath, casapath, carta, sourcetag, workingdir, vis, nvis], 
            open(workingdir+'/'+sourcetag+'/dirvises.npy', 'wb'), protocol=2)

#Save imaging parameters
pickle.dump([sourcetag,workingdir,vis,nvis,mosaic,phasecenter,weighting,robust,uvtaper,interactive], 
            open(workingdir+'/'+sourcetag+'/imaging/imagepars.npy', 'wb'), protocol=2)

Convert visibilities in CASA MS format to a python save file. This is done with CASA within the notebook. 

In [4]:
os.chdir('calibratedms')
!{casapath}/casa -c mstonumpyortxt_multiple.py


optional configuration file config.py not found, continuing CASA startup without it

]0;IPython: hd377/calibratedmsNo event loop hook running.
Using matplotlib backend: <object object at 0x10f862960>
CASA 6.6.1.17 -- Common Astronomy Software Applications [6.6.1.17]
['hd377_calibrated_merged.ms']
['hd377_calibrated_merged.ms']
Found data with 36396 uv points per channel
with 1 channels per SPW and 2 polarizations,
12 SPWs and Channel 0 frequency of 1st SPW of 232.97455884866974 GHz
corresponding to 1.2867928647725466 mm
Datasets has baselines between 8.371399028583255 and 43.50746241055415 m


Read in visibilities into Python, and use Galario's tool to figure out ideal cell size and image size.
Typically half the suggested image size is OK, and can save significant computation time.

In [5]:
from galario.double import get_image_size

#Read visibility data into python, figure out optimal pixel and image size for imaging. 
u=[[] for x in vis]
v=[[] for x in vis]
Re=[[] for x in vis]
Im=[[] for x in vis]
w=[[] for x in vis]
nxy=[[] for x in vis]
dxy=[[] for x in vis]
for i in np.arange(nvis):
    u[i], v[i], Re[i], Im[i], w[i] = np.load(vis[i][:-3]+'.npy')
    nxy[i], dxy[i] = get_image_size(u[i], v[i])
    nxy[i]/=2
    print('Pixel size (arcsec) and number of pixels required for dataset '+vis[i]+':')
    print(dxy[i]*180.0/np.pi*3600, nxy[i])    

#Figure out pix size and number for concatenated image. This is needed if more than one visibility dataset is present.
dxyall=np.min(dxy)
nxyall=int(np.ceil(np.max(np.asarray(dxy)*np.asarray(nxy))/dxyall/2.0)*2)
print('')
print('*** Pixel size (arcsec) and number of pixels that will be used: ***')
print(dxyall*180.0/np.pi*3600, nxyall)    

#Save pixel sizes and number of pixels, for imaging and later fitting
pickle.dump([dxyall, int(nxyall)], open(workingdir+'/'+sourcetag+'/calibratedms/pixinfo.npy', 'wb'), protocol=2)

Pixel size (arcsec) and number of pixels required for dataset hd377_calibrated_merged.ms:
0.7509504319361275 64.0

*** Pixel size (arcsec) and number of pixels that will be used: ***
0.7509504319361275 64


Then run the CLEANing using CASA's tclean, also producing FITS files for the image and primary beam, which we will need later, under the hood. For multiple visibility datasets, this produces images for each dataset AND for the concatenated dataset. 

Note that there may well be large weight differentials between datasets, especially if some come from early ALMA Cycles, or other telescope facilities. These are not accounted for during any concatenation that happens here, but will be accounted for in post-processing after visibility fitting, as the weights are (somewhat) fitted for in the modelling.

In [6]:
os.chdir('../imaging')

!{casapath}/casa -c imagingscript_multiple.py


optional configuration file config.py not found, continuing CASA startup without it

]0;IPython: hd377/imagingNo event loop hook running.
Using matplotlib backend: <object object at 0x107d23960>
CASA 6.6.1.17 -- Common Astronomy Software Applications [6.6.1.17]
2025-08-29 04:00:53	WARN	task_tclean::SIImageStore::restore (file /Users/casaci/bamboohome/xml-data/build-dir/CASA-CRBC661-BPOSX12P38/casa6/casatools/src/code/synthesis/ImagerObjects/SIImageStore.cc, line 2284)	Restoring with an empty model image. Only residuals will be processed to form the output restored image.


Nice dataset! You now have CLEAN images which we will plot nicely with matplotlib later during postprocessing.

For now, take a look at your image with the CASA viewer (command below), measure the RMS noise level in an empty region and note it for later.
You may now proceed with the visibility fitting in the vismodelling tutorial :)

In [7]:
### Open fits images in CARTA
filefront = vis[0].split('/')[-1].split('.')[0]
os.chdir(workingdir + sourcetag + '/imaging/')
#print(workingdir + sourcetag + '/imaging/')
#print(filefront+'_'+weighting+'_pb.fits')
#print(filefront+'_'+weighting+'.fits')

!{carta + ' ' + filefront+'_'+weighting+'.fits'} #Output image

CARTA will use the default ephemerides and geodetic data.

[2025-08-29 04:00:53.904Z] [CARTA] [info] Writing to the log file: /Users/jonty/.carta/log/carta.log
[2025-08-29 04:00:53.904Z] [CARTA] [info] /Applications/CARTA.app/Contents/Resources/app/carta-backend/bin/carta_backend: Version 5.0.3

[2025-08-29 04:00:53.906Z] [CARTA] [info] Serving CARTA frontend from /Applications/CARTA.app/Contents/Resources/app

[2025-08-29 04:00:53.907Z] [CARTA] [info] Listening on port 18535 with top level folder /, starting folder /Users/jonty/mydata/aca_disks/hd377/imaging. The number of OpenMP worker threads will be handled automatically.
[2025-08-29 04:00:53.907Z] [CARTA] [info] CARTA is accessible at http://localhost:18535/?token=603f3833-d629-4862-8b80-61fe963f5d15

[2025-08-29 04:00:54.753Z] [CARTA] [info] 0x7fbda67047c0 ::Session (294152349:1)
[2025-08-29 04:00:54.753Z] [CARTA] [info] Session 294152349 [127.0.0.1] Connected. Num sessions: 1

